# Faster R-CNN From Scratch - Seed Germination

Notebook này train **Faster R-CNN từ đầu** cho bài toán phát hiện hạt và phân loại trạng thái `non_germinated` / `germinated` trên ảnh Petri gốc.

Điểm quan trọng:
- Không dùng pretrained weights.
- Không dùng `crops.zip`.
- Chỉ cần đưa `GermPredDataset.zip` lên Google Drive hoặc upload trực tiếp vào Colab.
- Notebook tự giải nén, parse XML Pascal VOC, chia train/val/test theo `sequence_id`, train model, đánh giá và xuất file zip kết quả.

Dataset zip kỳ vọng chứa cấu trúc:

```text
GermPredDataset/
├── PennisetumGlaucum/
│   ├── img/
│   └── true_ann/
├── SecaleCereale/
│   ├── img/
│   └── true_ann/
└── ZeaMays/
    ├── img/
    └── true_ann/
```


## Cell 0 - Mount Google Drive

Nếu bạn để `GermPredDataset.zip` trên Drive, chạy cell này trước. Notebook sẽ tự tìm các vị trí phổ biến như:

```text
/content/drive/MyDrive/GermPredDataset.zip
/content/drive/MyDrive/SeedGermination/GermPredDataset.zip
```

Nếu upload zip trực tiếp vào `/content`, cell mount Drive có thể bỏ qua.


In [ ]:
# Optional but recommended when GermPredDataset.zip is on Google Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('Not running on Google Colab. Skip Drive mount.')


## Cell 1 - Kiểm tra môi trường

In [ ]:
import os
import sys
from pathlib import Path

import torch
import torchvision

print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    print('CUDA capability:', torch.cuda.get_device_capability(0))

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision('high')
        print("TF32 matmul precision enabled ('high').")
    except Exception as exc:
        print('TF32 setup skipped:', exc)


## Cell 1.5 - Cài dependency đánh giá mAP nếu thiếu

Colab thường có sẵn PyTorch/torchvision nhưng có thể thiếu `torchmetrics` hoặc `pycocotools`. Cell này chỉ cài khi thiếu.

In [ ]:
import importlib.util
import subprocess
import sys

missing_packages = []
if importlib.util.find_spec('torchmetrics') is None:
    missing_packages.append('torchmetrics')
if importlib.util.find_spec('pycocotools') is None:
    missing_packages.append('pycocotools')

if missing_packages:
    print('Installing missing packages:', missing_packages)
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *missing_packages])
else:
    print('torchmetrics and pycocotools are available.')


## Cell 2 - Import thư viện và seed

In [ ]:
import csv
import json
import math
import random
import shutil
import time
import zipfile
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw, ImageFont
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as F
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Seed:', SEED)


## Cell 3 - Cấu hình chính

Notebook baseline Custom CNN dùng `BATCH_SIZE = 256` vì đó là crop classification 224x224. Faster R-CNN train trên ảnh gốc 640px và lưu nhiều proposal/feature map, nên batch lớn thực tế trên A100 là khoảng `8-16`. Mặc định notebook này dùng `BATCH_SIZE = 8`; nếu A100 80GB còn dư VRAM, có thể thử `12` hoặc `16`.


In [ ]:
# Dataset paths.
DATASET_ROOT = Path('/content/data/raw/GermPredDataset')
LOCAL_DATASET_ROOT = Path('/Users/syhung/Seed-Germination-Monitoring-System/data/raw/GermPredDataset')

# Nếu zip của bạn nằm chỗ khác trên Drive, sửa dòng này.
USER_ZIP_PATH = None  # ví dụ: Path('/content/drive/MyDrive/SeedGermination/GermPredDataset.zip')

ZIP_CANDIDATES = [
    Path('/content/GermPredDataset.zip'),
    Path('/content/drive/MyDrive/GermPredDataset.zip'),
    Path('/content/drive/MyDrive/SeedGermination/GermPredDataset.zip'),
    Path('/content/drive/MyDrive/datasets/GermPredDataset.zip'),
]
if USER_ZIP_PATH is not None:
    ZIP_CANDIDATES.insert(0, Path(USER_ZIP_PATH))

# Detection classes: 0 is reserved for background by Faster R-CNN.
NUM_CLASSES = 3
DETECTION_CLASS_NAMES = {
    0: 'background',
    1: 'non_germinated',
    2: 'germinated',
}
ID_TO_COLOR = {
    1: 'dodgerblue',
    2: 'lime',
}

# Training config.
# A100-friendly default. Faster R-CNN is much heavier than crop classification,
# so do not copy the baseline classification batch size of 256 here.
# If your Colab A100 has 80GB VRAM and memory is still low, try 12 or 16.
BATCH_SIZE = 8
NUM_WORKERS = min(4, os.cpu_count() or 4)
EPOCHS = 25
LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005
STEP_SIZE = 8
LR_GAMMA = 0.1

# Faster R-CNN image resize. Raw images are around Petri-frame scale; 640 is a safe start.
MIN_SIZE = 640
MAX_SIZE = 640

# Development switches.
USE_SUBSET = False
MAX_TRAIN_IMAGES = 800
MAX_VAL_IMAGES = 200
MAX_TEST_IMAGES = 200
VAL_EVAL_MAX_IMAGES = 500  # 0 means full validation set each epoch.
TEST_EVAL_MAX_IMAGES = 0   # 0 means full test set after training.
CONFIDENCE_THRESHOLD = 0.50
NUM_PREDICTION_SAMPLES = 12

# Mixed precision.
USE_AMP = torch.cuda.is_available()
AMP_DTYPE = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16

# Output root. Drive is preferred so outputs survive Colab runtime reset.
if Path('/content/drive/MyDrive').exists():
    OUTPUT_ROOT = Path('/content/drive/MyDrive/SeedGermination/faster_rcnn_scratch_outputs')
elif Path('/content').exists():
    OUTPUT_ROOT = Path('/content/outputs/faster_rcnn_scratch')
else:
    OUTPUT_ROOT = Path('outputs/faster_rcnn_scratch')

CHECKPOINT_DIR = OUTPUT_ROOT / 'checkpoints'
LOG_DIR = OUTPUT_ROOT / 'logs'
REPORT_DIR = OUTPUT_ROOT / 'reports'
FIGURE_DIR = OUTPUT_ROOT / 'figures'
PREDICTION_DIR = OUTPUT_ROOT / 'predictions'
for path in [CHECKPOINT_DIR, LOG_DIR, REPORT_DIR, FIGURE_DIR, PREDICTION_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print('BATCH_SIZE:', BATCH_SIZE)
print('EPOCHS:', EPOCHS)
print('OUTPUT_ROOT:', OUTPUT_ROOT)
print('USE_AMP:', USE_AMP, 'AMP_DTYPE:', AMP_DTYPE)
print('NO PRETRAINED WEIGHTS USED: weights=None, weights_backbone=None')


## Cell 4 - Tìm và giải nén `GermPredDataset.zip`

In [ ]:
SPECIES_FOLDERS = ['PennisetumGlaucum', 'SecaleCereale', 'ZeaMays']
IMAGE_EXTENSIONS = ('.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff')


def has_dataset_layout(root: Path) -> bool:
    return all((root / species / 'img').is_dir() and (root / species / 'true_ann').is_dir() for species in SPECIES_FOLDERS)


def find_existing_dataset_root() -> Optional[Path]:
    candidates = [DATASET_ROOT, LOCAL_DATASET_ROOT, Path('data/raw/GermPredDataset')]
    for candidate in candidates:
        if has_dataset_layout(candidate):
            return candidate.resolve()
    return None


def find_zip_path() -> Optional[Path]:
    for candidate in ZIP_CANDIDATES:
        if candidate.exists():
            return candidate
    return None


def assert_zip_ready(zip_path: Path) -> None:
    if not zip_path.exists():
        raise FileNotFoundError(f'Không tìm thấy zip: {zip_path}')
    if not zipfile.is_zipfile(zip_path):
        size1 = zip_path.stat().st_size
        time.sleep(2)
        size2 = zip_path.stat().st_size
        if size1 != size2:
            raise RuntimeError('File zip có vẻ vẫn đang upload. Đợi upload xong rồi chạy lại cell.')
        raise RuntimeError(f'File không phải zip hợp lệ hoặc bị hỏng: {zip_path}')
    with zipfile.ZipFile(zip_path) as zf:
        bad_file = zf.testzip()
        if bad_file is not None:
            raise RuntimeError(f'Zip bị lỗi CRC/Header tại file: {bad_file}')


def extract_germpred_zip(zip_path: Path, target_root: Path) -> Path:
    assert_zip_ready(zip_path)
    target_root.parent.mkdir(parents=True, exist_ok=True)

    if has_dataset_layout(target_root):
        print('Dataset already extracted:', target_root)
        return target_root

    print('Extracting:', zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        names = zf.namelist()
        has_named_root = any(name.startswith('GermPredDataset/') for name in names)
        has_species_at_top = any(
            name.startswith(f'{species}/')
            for species in SPECIES_FOLDERS
            for name in names
        )

        if has_named_root:
            extract_dir = target_root.parent
        elif has_species_at_top:
            extract_dir = target_root
        else:
            extract_dir = target_root.parent

        extract_dir.mkdir(parents=True, exist_ok=True)
        zf.extractall(extract_dir)

    if has_dataset_layout(target_root):
        print('Extracted dataset root:', target_root)
        return target_root

    # Some archives contain one extra top-level folder. Search for the valid layout.
    for candidate in target_root.parent.rglob('GermPredDataset'):
        if has_dataset_layout(candidate):
            print('Found dataset root:', candidate)
            return candidate

    for candidate in target_root.parent.iterdir():
        if candidate.is_dir() and has_dataset_layout(candidate):
            print('Found dataset root:', candidate)
            return candidate

    raise FileNotFoundError('Đã giải nén nhưng không tìm thấy cấu trúc GermPredDataset hợp lệ.')


existing_root = find_existing_dataset_root()
if existing_root is not None:
    RAW_ROOT = existing_root
    print('Using existing dataset root:', RAW_ROOT)
else:
    zip_path = find_zip_path()
    if zip_path is None:
        raise FileNotFoundError(
            'Không tìm thấy GermPredDataset.zip. Hãy upload vào /content hoặc Google Drive.\n'
            'Các vị trí đang kiểm tra:\n' + '\n'.join(str(path) for path in ZIP_CANDIDATES)
        )
    RAW_ROOT = extract_germpred_zip(zip_path, DATASET_ROOT).resolve()

print('RAW_ROOT:', RAW_ROOT)


## Cell 5 - Validate raw dataset

In [ ]:
def count_files(root: Path):
    rows = []
    for species in SPECIES_FOLDERS:
        img_dir = root / species / 'img'
        ann_dir = root / species / 'true_ann'
        image_count = sum(1 for path in img_dir.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
        xml_count = sum(1 for path in ann_dir.iterdir() if path.suffix.lower() == '.xml')
        rows.append({'species': species, 'images': image_count, 'xml': xml_count})
    return pd.DataFrame(rows)

counts_df = count_files(RAW_ROOT)
display(counts_df)

missing = counts_df[(counts_df['images'] == 0) | (counts_df['xml'] == 0)]
if not missing.empty:
    raise RuntimeError('Dataset thiếu ảnh hoặc XML ở ít nhất một loài.')

if not (counts_df['images'] == counts_df['xml']).all():
    print('Cảnh báo: số ảnh và XML không bằng nhau ở ít nhất một loài. Parser sẽ báo lỗi chi tiết nếu thiếu cặp.')

print('Total images:', int(counts_df['images'].sum()))
print('Total XML:', int(counts_df['xml'].sum()))


## Cell 6 - Parse XML Pascal VOC và tạo records

In [ ]:
SPECIES_BY_FOLDER = {
    'PennisetumGlaucum': {'species_code': 'pg', 'species_name': 'Pennisetum glaucum'},
    'SecaleCereale': {'species_code': 'sc', 'species_name': 'Secale cereale'},
    'ZeaMays': {'species_code': 'zm', 'species_name': 'Zea mays'},
}

RAW_TO_DETECTION_LABEL = {
    'pg_im': 1, 'sc_im': 1, 'zm_im': 1,
    'pg_el': 2, 'sc_el': 2, 'zm_el': 2,
}
RAW_TO_BINARY_LABEL = {
    'pg_im': 'non_germinated', 'sc_im': 'non_germinated', 'zm_im': 'non_germinated',
    'pg_el': 'germinated', 'sc_el': 'germinated', 'zm_el': 'germinated',
}


def text_of(elem: Optional[ET.Element]) -> str:
    return elem.text.strip() if elem is not None and elem.text else ''


def parse_filename(file_stem: str) -> Optional[dict]:
    # Example: zm1_10_img001 -> species=zm, experiment=1, dish=10, frame=001, sequence=zm1_10
    if '_img' not in file_stem:
        return None
    left, frame_text = file_stem.rsplit('_img', maxsplit=1)
    if '_' not in left or not frame_text.isdigit():
        return None
    species_and_exp, dish_text = left.split('_', maxsplit=1)
    species_code = ''.join(ch for ch in species_and_exp if ch.isalpha()).lower()
    exp_text = species_and_exp[len(species_code):]
    if not species_code or not exp_text.isdigit() or not dish_text.isdigit():
        return None
    return {
        'species_code_from_filename': species_code,
        'experiment_id': int(exp_text),
        'dish_id': int(dish_text),
        'frame_id': int(frame_text),
        'sequence_id': f'{species_code}{int(exp_text)}_{int(dish_text)}',
    }


def find_image(img_dir: Path, file_stem: str) -> Optional[Path]:
    for ext in IMAGE_EXTENSIONS:
        candidate = img_dir / f'{file_stem}{ext}'
        if candidate.exists():
            return candidate
    return None


def read_xml_size(root: ET.Element) -> tuple[int, int]:
    size = root.find('size')
    if size is None:
        raise ValueError('missing <size>')
    width = int(float(text_of(size.find('width'))))
    height = int(float(text_of(size.find('height'))))
    return width, height


def read_bbox(obj: ET.Element) -> tuple[int, int, int, int]:
    box = obj.find('bndbox')
    if box is None:
        raise ValueError('missing <bndbox>')
    xmin = round(float(text_of(box.find('xmin'))))
    ymin = round(float(text_of(box.find('ymin'))))
    xmax = round(float(text_of(box.find('xmax'))))
    ymax = round(float(text_of(box.find('ymax'))))
    return xmin, ymin, xmax, ymax


def parse_germpred_dataset(raw_root: Path, max_xml_per_species: int = 0):
    records = []
    errors = []
    image_id = 1

    for species_folder in SPECIES_FOLDERS:
        species_dir = raw_root / species_folder
        img_dir = species_dir / 'img'
        ann_dir = species_dir / 'true_ann'
        species_info = SPECIES_BY_FOLDER[species_folder]
        xml_paths = sorted(ann_dir.glob('*.xml'))
        if max_xml_per_species > 0:
            xml_paths = xml_paths[:max_xml_per_species]
        print(f'Scanning {species_folder}: {len(xml_paths)} XML files')

        for xml_path in tqdm(xml_paths, leave=False):
            file_stem = xml_path.stem
            parsed_name = parse_filename(file_stem)
            if parsed_name is None:
                errors.append({'xml_path': str(xml_path), 'error': 'filename_parse_error', 'detail': file_stem})
                continue

            image_path = find_image(img_dir, file_stem)
            if image_path is None:
                errors.append({'xml_path': str(xml_path), 'error': 'missing_image', 'detail': file_stem})
                continue

            try:
                xml_root = ET.parse(xml_path).getroot()
                width, height = read_xml_size(xml_root)
            except Exception as exc:
                errors.append({'xml_path': str(xml_path), 'error': 'xml_read_error', 'detail': str(exc)})
                continue

            boxes = []
            labels = []
            raw_labels = []
            binary_labels = []

            for object_id, obj in enumerate(xml_root.findall('object')):
                raw_label = text_of(obj.find('name'))
                if raw_label not in RAW_TO_DETECTION_LABEL:
                    errors.append({'xml_path': str(xml_path), 'error': 'unknown_label', 'detail': raw_label})
                    continue
                try:
                    xmin, ymin, xmax, ymax = read_bbox(obj)
                except Exception as exc:
                    errors.append({'xml_path': str(xml_path), 'error': 'bbox_read_error', 'detail': f'object_id={object_id}, {exc}'})
                    continue
                if xmin >= xmax or ymin >= ymax:
                    errors.append({'xml_path': str(xml_path), 'error': 'invalid_bbox_order', 'detail': str((xmin, ymin, xmax, ymax))})
                    continue
                if xmin < 0 or ymin < 0 or xmax > width or ymax > height:
                    errors.append({'xml_path': str(xml_path), 'error': 'bbox_out_of_bounds', 'detail': f'{(xmin, ymin, xmax, ymax)}, size={(width, height)}'})
                    continue

                boxes.append([float(xmin), float(ymin), float(xmax), float(ymax)])
                labels.append(int(RAW_TO_DETECTION_LABEL[raw_label]))
                raw_labels.append(raw_label)
                binary_labels.append(RAW_TO_BINARY_LABEL[raw_label])

            if not boxes:
                errors.append({'xml_path': str(xml_path), 'error': 'empty_valid_objects', 'detail': ''})
                continue

            records.append({
                'image_id': image_id,
                'image_path': image_path,
                'xml_path': xml_path,
                'filename': image_path.name,
                'file_stem': file_stem,
                'species_folder': species_folder,
                'species_code': species_info['species_code'],
                'species_name': species_info['species_name'],
                'sequence_id': parsed_name['sequence_id'],
                'frame_id': parsed_name['frame_id'],
                'width': width,
                'height': height,
                'boxes': boxes,
                'labels': labels,
                'raw_labels': raw_labels,
                'binary_labels': binary_labels,
            })
            image_id += 1

    return records, errors


# Always parse all XML first so sequence split remains representative.
# USE_SUBSET only limits train/val/test image lists after split assignment.
max_xml = 0
if USE_SUBSET:
    print('USE_SUBSET=True: full XML metadata will still be parsed; train/val/test image lists are limited after splitting.')

records, parse_errors = parse_germpred_dataset(RAW_ROOT, max_xml_per_species=max_xml)
print('Valid image records:', len(records))
print('Parse errors:', len(parse_errors))

if parse_errors:
    error_df = pd.DataFrame(parse_errors)
    display(error_df.head(20))
    error_df.to_csv(REPORT_DIR / 'parse_errors.csv', index=False)

if not records:
    raise RuntimeError('Không parse được image record hợp lệ nào.')

object_count = sum(len(row['boxes']) for row in records)
print('Valid objects / boxes:', object_count)


## Cell 7 - Chia train/val/test theo `sequence_id`

In [ ]:
def make_sequence_summary(records: list[dict]) -> list[dict]:
    grouped = defaultdict(list)
    for row in records:
        grouped[row['sequence_id']].append(row)

    sequence_rows = []
    for sequence_id, group in grouped.items():
        first = group[0]
        label_counts = Counter(label for image in group for label in image['binary_labels'])
        sequence_rows.append({
            'sequence_id': sequence_id,
            'species_code': first['species_code'],
            'species_name': first['species_name'],
            'num_images': len(group),
            'num_objects': sum(len(image['boxes']) for image in group),
            'germinated_count': label_counts.get('germinated', 0),
            'non_germinated_count': label_counts.get('non_germinated', 0),
        })
    for row in sequence_rows:
        total = row['num_objects']
        row['germinated_ratio'] = row['germinated_count'] / total if total else 0.0
    return sequence_rows


def assign_sequence_splits(sequence_rows: list[dict], seed: int = 42, train_ratio: float = 0.80, val_ratio: float = 0.10):
    rng = random.Random(seed)
    by_species = defaultdict(list)
    for row in sequence_rows:
        by_species[row['species_code']].append(dict(row))

    split_rows = []
    for species_code in sorted(by_species):
        species_rows = by_species[species_code]
        rng.shuffle(species_rows)
        n = len(species_rows)
        n_train = round(n * train_ratio)
        n_val = round(n * val_ratio)
        for index, row in enumerate(species_rows):
            if index < n_train:
                split = 'train'
            elif index < n_train + n_val:
                split = 'val'
            else:
                split = 'test'
            row['split'] = split
            split_rows.append(row)
        print(f'{species_code}: total={n}, train={n_train}, val={n_val}, test={n - n_train - n_val}')
    return split_rows


def load_existing_sequence_split(sequence_ids: set[str]) -> Optional[dict[str, str]]:
    candidates = [
        Path('/content/data/metadata/sequence_split.csv'),
        Path('data/metadata/sequence_split.csv'),
        Path('/Users/syhung/Seed-Germination-Monitoring-System/data/metadata/sequence_split.csv'),
    ]
    for csv_path in candidates:
        if not csv_path.exists():
            continue
        df = pd.read_csv(csv_path)
        if {'sequence_id', 'split'} - set(df.columns):
            continue
        split_map = dict(zip(df['sequence_id'].astype(str), df['split'].astype(str)))
        missing = sequence_ids - set(split_map)
        if not missing:
            print('Using existing sequence split:', csv_path)
            return split_map
        print(f'Existing split {csv_path} missing {len(missing)} sequences; generating a new split.')
    return None


sequence_rows = make_sequence_summary(records)
sequence_ids = {row['sequence_id'] for row in sequence_rows}
existing_split_map = load_existing_sequence_split(sequence_ids)

if existing_split_map is None:
    split_sequence_rows = assign_sequence_splits(sequence_rows, seed=SEED)
    split_map = {row['sequence_id']: row['split'] for row in split_sequence_rows}
else:
    split_map = existing_split_map
    split_sequence_rows = [dict(row, split=split_map[row['sequence_id']]) for row in sequence_rows]

for row in records:
    row['split'] = split_map[row['sequence_id']]

# Leakage check.
leakage = defaultdict(set)
for row in records:
    leakage[row['sequence_id']].add(row['split'])
leaked = {sequence_id: splits for sequence_id, splits in leakage.items() if len(splits) > 1}
if leaked:
    raise RuntimeError(f'Data leakage detected: {list(leaked.items())[:5]}')
print('Leakage check: PASSED')

sequence_df = pd.DataFrame(split_sequence_rows).sort_values(['species_code', 'sequence_id'])
image_split_df = pd.DataFrame([
    {
        'image_id': row['image_id'],
        'filename': row['filename'],
        'sequence_id': row['sequence_id'],
        'species_code': row['species_code'],
        'split': row['split'],
        'num_objects': len(row['boxes']),
    }
    for row in records
])

sequence_df.to_csv(REPORT_DIR / 'sequence_split.csv', index=False)
image_split_df.to_csv(REPORT_DIR / 'image_split.csv', index=False)

split_summary = image_split_df.groupby('split').agg(images=('filename', 'count'), objects=('num_objects', 'sum')).reindex(['train', 'val', 'test'])
display(split_summary)

display(sequence_df.groupby(['split', 'species_code']).size().unstack(fill_value=0).reindex(['train', 'val', 'test']))


## Cell 8 - PyTorch DetectionDataset và DataLoader

In [ ]:
class GerminationDetectionDataset(Dataset):
    def __init__(self, records: list[dict], train: bool = False):
        self.records = records
        self.train = train

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        image = Image.open(record['image_path']).convert('RGB')
        image_tensor = F.to_tensor(image)

        boxes = torch.as_tensor(record['boxes'], dtype=torch.float32)
        labels = torch.as_tensor(record['labels'], dtype=torch.int64)

        if self.train:
            if random.random() < 0.5:
                image_tensor = torch.flip(image_tensor, dims=[2])
                width = record['width']
                xmin = boxes[:, 0].clone()
                xmax = boxes[:, 2].clone()
                boxes[:, 0] = width - xmax
                boxes[:, 2] = width - xmin
            if random.random() < 0.5:
                image_tensor = torch.flip(image_tensor, dims=[1])
                height = record['height']
                ymin = boxes[:, 1].clone()
                ymax = boxes[:, 3].clone()
                boxes[:, 1] = height - ymax
                boxes[:, 3] = height - ymin

        area = (boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])
        iscrowd = torch.zeros((boxes.shape[0],), dtype=torch.int64)
        image_id = torch.tensor([record['image_id']], dtype=torch.int64)

        target = {
            'boxes': boxes,
            'labels': labels,
            'image_id': image_id,
            'area': area,
            'iscrowd': iscrowd,
        }
        return image_tensor, target


def collate_fn(batch):
    return tuple(zip(*batch))


def maybe_subset(rows: list[dict], max_images: int) -> list[dict]:
    if not USE_SUBSET or max_images <= 0 or len(rows) <= max_images:
        return rows
    rng = random.Random(SEED)
    selected = rows.copy()
    rng.shuffle(selected)
    return selected[:max_images]


train_records = [row for row in records if row['split'] == 'train']
val_records = [row for row in records if row['split'] == 'val']
test_records = [row for row in records if row['split'] == 'test']

train_records = maybe_subset(train_records, MAX_TRAIN_IMAGES)
val_records = maybe_subset(val_records, MAX_VAL_IMAGES)
test_records = maybe_subset(test_records, MAX_TEST_IMAGES)

train_dataset = GerminationDetectionDataset(train_records, train=True)
val_dataset = GerminationDetectionDataset(val_records, train=False)
test_dataset = GerminationDetectionDataset(test_records, train=False)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=collate_fn,
    pin_memory=torch.cuda.is_available(),
    persistent_workers=NUM_WORKERS > 0,
)

print('Train images:', len(train_dataset))
print('Val images:', len(val_dataset))
print('Test images:', len(test_dataset))

sample_image, sample_target = train_dataset[0]
print('Sample image tensor:', tuple(sample_image.shape), sample_image.dtype, float(sample_image.min()), float(sample_image.max()))
print('Sample boxes:', sample_target['boxes'].shape)
print('Sample labels:', torch.unique(sample_target['labels']).tolist())


## Cell 9 - Smoke checks trước train

In [ ]:
def run_smoke_checks(dataset: GerminationDetectionDataset, name: str, max_items: int = 200) -> None:
    label_values = set()
    checked = min(len(dataset), max_items)
    for index in range(checked):
        image, target = dataset[index]
        _, height, width = image.shape
        boxes = target['boxes']
        labels = target['labels']
        if boxes.numel() == 0:
            raise RuntimeError(f'{name}[{index}] has no boxes')
        if not torch.all(boxes[:, 0] < boxes[:, 2]) or not torch.all(boxes[:, 1] < boxes[:, 3]):
            raise RuntimeError(f'{name}[{index}] has invalid bbox order')
        if boxes[:, 0].min() < 0 or boxes[:, 1].min() < 0 or boxes[:, 2].max() > width or boxes[:, 3].max() > height:
            raise RuntimeError(f'{name}[{index}] has out-of-bounds boxes')
        label_values.update(labels.tolist())
    invalid_labels = label_values - {1, 2}
    if invalid_labels:
        raise RuntimeError(f'{name} has invalid labels: {invalid_labels}')
    print(f'{name}: smoke check passed on {checked} images. Labels={sorted(label_values)}')

run_smoke_checks(train_dataset, 'train')
run_smoke_checks(val_dataset, 'val')
run_smoke_checks(test_dataset, 'test')


## Cell 10 - Tạo Faster R-CNN không pretrained

In [ ]:
def build_faster_rcnn_scratch(num_classes: int = NUM_CLASSES):
    try:
        model = fasterrcnn_resnet50_fpn(
            weights=None,
            weights_backbone=None,
            num_classes=num_classes,
            min_size=MIN_SIZE,
            max_size=MAX_SIZE,
        )
    except TypeError:
        # Compatibility for older torchvision versions.
        model = fasterrcnn_resnet50_fpn(
            pretrained=False,
            pretrained_backbone=False,
            num_classes=num_classes,
            min_size=MIN_SIZE,
            max_size=MAX_SIZE,
        )
    return model


model = build_faster_rcnn_scratch().to(DEVICE)
print(model.__class__.__name__)
print('NO PRETRAINED WEIGHTS USED')
print('Trainable parameters:', sum(p.numel() for p in model.parameters() if p.requires_grad))


## Cell 11 - Hàm train và evaluate mAP

In [ ]:
def move_targets_to_device(targets, device):
    moved = []
    for target in targets:
        moved.append({key: value.to(device) if torch.is_tensor(value) else value for key, value in target.items()})
    return moved


def train_one_epoch(model, optimizer, data_loader, epoch: int):
    model.train()
    total_loss = 0.0
    loss_sums = defaultdict(float)
    total_batches = 0
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP and AMP_DTYPE == torch.float16)

    progress = tqdm(data_loader, desc=f'Epoch {epoch} train', leave=False)
    for images, targets in progress:
        images = [image.to(DEVICE, non_blocking=True) for image in images]
        targets = move_targets_to_device(targets, DEVICE)

        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type='cuda', dtype=AMP_DTYPE, enabled=USE_AMP):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())

        if scaler.is_enabled():
            scaler.scale(losses).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            losses.backward()
            optimizer.step()

        batch_loss = float(losses.detach().cpu())
        total_loss += batch_loss
        total_batches += 1
        for key, value in loss_dict.items():
            loss_sums[key] += float(value.detach().cpu())
        progress.set_postfix(loss=f'{batch_loss:.4f}')

    metrics = {'train_loss': total_loss / max(total_batches, 1)}
    for key, value in loss_sums.items():
        metrics[key] = value / max(total_batches, 1)
    return metrics


def evaluate_map(model, data_loader, max_images: int = 0, split_name: str = 'val'):
    try:
        from torchmetrics.detection.mean_ap import MeanAveragePrecision
    except Exception as exc:
        print(f'torchmetrics MeanAveragePrecision unavailable: {exc}')
        return {'map': None, 'map_50': None, 'map_75': None, 'evaluated_images': 0, 'note': 'mAP unavailable'}

    metric = MeanAveragePrecision(box_format='xyxy', class_metrics=True)
    model.eval()
    evaluated = 0
    with torch.no_grad():
        for images, targets in tqdm(data_loader, desc=f'Evaluate {split_name}', leave=False):
            images_on_device = [image.to(DEVICE, non_blocking=True) for image in images]
            outputs = model(images_on_device)

            preds = []
            target_list = []
            for output, target in zip(outputs, targets):
                preds.append({
                    'boxes': output['boxes'].detach().cpu(),
                    'scores': output['scores'].detach().cpu(),
                    'labels': output['labels'].detach().cpu(),
                })
                target_list.append({
                    'boxes': target['boxes'].detach().cpu(),
                    'labels': target['labels'].detach().cpu(),
                })
            metric.update(preds, target_list)
            evaluated += len(images)
            if max_images and evaluated >= max_images:
                break

    result = metric.compute()
    def to_float(value):
        if torch.is_tensor(value):
            if value.numel() == 1:
                return float(value.cpu())
            return [float(item) for item in value.cpu().flatten()]
        return value

    metrics = {key: to_float(value) for key, value in result.items()}
    metrics['evaluated_images'] = evaluated
    return metrics


optimizer = torch.optim.SGD(
    model.parameters(),
    lr=LEARNING_RATE,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
)
lr_scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=STEP_SIZE, gamma=LR_GAMMA)
print(optimizer)


## Cell 12 - Train model

Cell này sẽ mất thời gian. Với A100, hãy bắt đầu bằng `BATCH_SIZE = 8`. Nếu Colab báo còn dư nhiều VRAM, lần sau có thể tăng lên `12` hoặc `16`.


In [ ]:
history = []
best_score = -1.0
best_checkpoint_path = CHECKPOINT_DIR / 'best_faster_rcnn_resnet50_fpn_scratch.pth'
last_checkpoint_path = CHECKPOINT_DIR / 'last_faster_rcnn_resnet50_fpn_scratch.pth'

start_time = time.time()
for epoch in range(1, EPOCHS + 1):
    epoch_start = time.time()
    train_metrics = train_one_epoch(model, optimizer, train_loader, epoch)
    lr_scheduler.step()

    val_metrics = evaluate_map(model, val_loader, max_images=VAL_EVAL_MAX_IMAGES, split_name='val')
    val_map = val_metrics.get('map')
    val_map_50 = val_metrics.get('map_50')
    score = val_map_50 if isinstance(val_map_50, float) else (val_map if isinstance(val_map, float) else -train_metrics['train_loss'])

    row = {
        'epoch': epoch,
        'lr': optimizer.param_groups[0]['lr'],
        'elapsed_min': (time.time() - epoch_start) / 60,
        **train_metrics,
        'val_map': val_map,
        'val_map_50': val_map_50,
        'val_map_75': val_metrics.get('map_75'),
        'val_evaluated_images': val_metrics.get('evaluated_images'),
    }
    history.append(row)
    pd.DataFrame(history).to_csv(LOG_DIR / 'training_history.csv', index=False)

    checkpoint = {
        'model_name': 'fasterrcnn_resnet50_fpn_scratch',
        'state_dict': model.state_dict(),
        'class_names': DETECTION_CLASS_NAMES,
        'config': {
            'num_classes': NUM_CLASSES,
            'min_size': MIN_SIZE,
            'max_size': MAX_SIZE,
            'batch_size': BATCH_SIZE,
            'epochs': EPOCHS,
            'learning_rate': LEARNING_RATE,
            'weights': None,
            'weights_backbone': None,
        },
        'epoch': epoch,
        'history': history,
        'val_metrics': val_metrics,
    }
    torch.save(checkpoint, last_checkpoint_path)

    if score > best_score:
        best_score = score
        torch.save(checkpoint, best_checkpoint_path)
        print(f'New best checkpoint at epoch {epoch}: score={best_score}')

    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train_loss={train_metrics['train_loss']:.4f} | "
        f"val_map={val_map} | val_map_50={val_map_50} | "
        f"time={row['elapsed_min']:.1f} min"
    )

train_time_min = (time.time() - start_time) / 60
print('Training finished. Minutes:', train_time_min)
print('Best checkpoint:', best_checkpoint_path)


## Cell 13 - Learning curves

In [ ]:
history_df = pd.DataFrame(history)
history_path = LOG_DIR / 'training_history.csv'
history_df.to_csv(history_path, index=False)
display(history_df.tail())

plt.figure(figsize=(10, 4))
plt.plot(history_df['epoch'], history_df['train_loss'], marker='o', label='train_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Faster R-CNN Scratch Training Loss')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
loss_fig = FIGURE_DIR / 'learning_curve_loss.png'
plt.savefig(loss_fig, dpi=160)
plt.show()

if 'val_map_50' in history_df.columns and history_df['val_map_50'].notna().any():
    plt.figure(figsize=(10, 4))
    plt.plot(history_df['epoch'], history_df['val_map_50'], marker='o', label='val_mAP50')
    if history_df['val_map'].notna().any():
        plt.plot(history_df['epoch'], history_df['val_map'], marker='o', label='val_mAP')
    plt.xlabel('Epoch')
    plt.ylabel('mAP')
    plt.title('Validation mAP')
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    map_fig = FIGURE_DIR / 'learning_curve_map.png'
    plt.savefig(map_fig, dpi=160)
    plt.show()

print('Saved:', history_path)


## Cell 14 - Test evaluation với checkpoint tốt nhất

In [ ]:
if best_checkpoint_path.exists():
    best_checkpoint = torch.load(best_checkpoint_path, map_location=DEVICE)
    model.load_state_dict(best_checkpoint['state_dict'])
    print('Loaded best checkpoint epoch:', best_checkpoint.get('epoch'))
else:
    print('Best checkpoint not found. Using current model state.')

test_metrics = evaluate_map(model, test_loader, max_images=TEST_EVAL_MAX_IMAGES, split_name='test')
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

class_rows = []
classes = test_metrics.get('classes')
map_per_class = test_metrics.get('map_per_class')
mar_100_per_class = test_metrics.get('mar_100_per_class')
if isinstance(classes, list) and isinstance(map_per_class, list):
    for index, class_id_value in enumerate(classes):
        class_id = int(class_id_value)
        if class_id == 0:
            continue
        class_rows.append({
            'class_id': class_id,
            'class_name': DETECTION_CLASS_NAMES.get(class_id, str(class_id)),
            'map': map_per_class[index] if index < len(map_per_class) else None,
            'mar_100': mar_100_per_class[index] if isinstance(mar_100_per_class, list) and index < len(mar_100_per_class) else None,
        })
class_metrics_df = pd.DataFrame(class_rows)
class_metrics_path = REPORT_DIR / 'class_metrics.csv'
class_metrics_df.to_csv(class_metrics_path, index=False)
print('Saved class metrics:', class_metrics_path)

summary = {
    'model': 'fasterrcnn_resnet50_fpn_scratch',
    'pretrained': False,
    'weights': None,
    'weights_backbone': None,
    'num_classes': NUM_CLASSES,
    'class_names': DETECTION_CLASS_NAMES,
    'train_images': len(train_dataset),
    'val_images': len(val_dataset),
    'test_images': len(test_dataset),
    'total_records': len(records),
    'total_objects': sum(len(row['boxes']) for row in records),
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'train_time_min': train_time_min,
    'best_epoch': int(best_checkpoint.get('epoch')) if best_checkpoint_path.exists() else None,
    'test_metrics': test_metrics,
}
summary_path = REPORT_DIR / 'test_summary.json'
with summary_path.open('w', encoding='utf-8') as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
print('Saved summary:', summary_path)


## Cell 15 - Lưu ảnh dự đoán mẫu

In [ ]:
def draw_predictions(image: Image.Image, output: dict, threshold: float = CONFIDENCE_THRESHOLD) -> Image.Image:
    image = image.copy()
    draw = ImageDraw.Draw(image)
    boxes = output['boxes'].detach().cpu()
    labels = output['labels'].detach().cpu()
    scores = output['scores'].detach().cpu()

    for box, label, score in zip(boxes, labels, scores):
        score_value = float(score)
        label_id = int(label)
        if score_value < threshold or label_id not in DETECTION_CLASS_NAMES:
            continue
        color = ID_TO_COLOR.get(label_id, 'yellow')
        x1, y1, x2, y2 = [float(v) for v in box]
        text = f"{DETECTION_CLASS_NAMES[label_id]} {score_value:.2f}"
        draw.rectangle([x1, y1, x2, y2], outline=color, width=2)
        text_box = draw.textbbox((x1, y1), text)
        draw.rectangle(text_box, fill=color)
        draw.text((x1, y1), text, fill='black')
    return image


model.eval()
PREDICTION_DIR.mkdir(parents=True, exist_ok=True)
sample_indices = np.linspace(0, max(len(test_records) - 1, 0), num=min(NUM_PREDICTION_SAMPLES, len(test_records)), dtype=int)

saved_images = []
with torch.no_grad():
    for idx in tqdm(sample_indices, desc='Saving prediction samples'):
        record = test_records[int(idx)]
        image = Image.open(record['image_path']).convert('RGB')
        image_tensor = F.to_tensor(image).to(DEVICE)
        output = model([image_tensor])[0]
        drawn = draw_predictions(image, output, threshold=CONFIDENCE_THRESHOLD)
        save_path = PREDICTION_DIR / f"{record['file_stem']}_prediction.jpg"
        drawn.save(save_path, quality=95)
        saved_images.append(save_path)

print('Saved prediction samples:', len(saved_images))
for path in saved_images[:5]:
    print(path)

if saved_images:
    display(Image.open(saved_images[0]))


## Cell 16 - Bảng so sánh report-ready

Lưu ý: Custom CNN và ResNet18 là **crop classification**, còn Faster R-CNN là **object detection**. Không nên so accuracy classification trực tiếp với mAP detection như cùng một loại metric.


In [ ]:
baseline_reference = {
    'method': 'Custom CNN baseline',
    'task_type': 'crop classification',
    'pretrained': False,
    'accuracy': 0.8336,
    'macro_f1': 0.8316,
    'map': None,
    'map_50': None,
    'note': 'Baseline ?? train tr?n crops.zip',
}
faster_rcnn_result = {
    'method': 'Faster R-CNN scratch',
    'task_type': 'object detection',
    'pretrained': False,
    'accuracy': None,
    'macro_f1': None,
    'map': test_metrics.get('map'),
    'map_50': test_metrics.get('map_50'),
    'note': 'Ph??ng ph?p ch?nh: train from scratch, raw image + XML bbox',
}
comparison_df = pd.DataFrame([baseline_reference, faster_rcnn_result])
comparison_path = REPORT_DIR / 'method_comparison.csv'
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)
print('Saved comparison:', comparison_path)


## Cell 17 - Export zip kết quả vào Drive và tải về

In [ ]:
if Path('/content/drive/MyDrive').exists():
    export_base = Path('/content/faster_rcnn_scratch_outputs_export')
    zip_base = Path('/content/drive/MyDrive/SeedGermination/faster_rcnn_scratch_outputs')
elif Path('/content').exists():
    export_base = Path('/content/faster_rcnn_scratch_outputs_export')
    zip_base = Path('/content/faster_rcnn_scratch_outputs')
else:
    export_base = Path('outputs/faster_rcnn_scratch_export')
    zip_base = Path('outputs/faster_rcnn_scratch_outputs')

# Copy selected artifacts to a temporary export folder first so download is simple.
export_base.parent.mkdir(parents=True, exist_ok=True)
zip_base.parent.mkdir(parents=True, exist_ok=True)
if export_base.exists():
    shutil.rmtree(export_base)
shutil.copytree(OUTPUT_ROOT, export_base)

archive_path = shutil.make_archive(str(zip_base), 'zip', root_dir=export_base)
print('Created archive:', archive_path)
if Path(archive_path).is_relative_to('/content/drive'):
    print('Saved zip to Google Drive:', archive_path)
print('Archive size MB:', Path(archive_path).stat().st_size / (1024 * 1024))

try:
    from google.colab import files
    files.download(archive_path)
    print('Download button requested in Colab.')
except ModuleNotFoundError:
    print('Not on Colab. Zip path:', archive_path)


## Cell 18 - Cách dùng sau khi train

Sau khi tải zip về, các file quan trọng nhất là:

```text
checkpoints/best_faster_rcnn_resnet50_fpn_scratch.pth
reports/test_summary.json
reports/method_comparison.csv
logs/training_history.csv
figures/*.png
predictions/*.jpg
```

Checkpoint này là model Faster R-CNN train từ đầu, không pretrained. Đây nên là model chính để demo nếu thầy không cho dùng pretrained.
